In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("C:/Users/Lenovo/Downloads/PL_team_matches.csv")
df["Date"] = pd.to_datetime(df["Date"])
df_home = df[df["HomeAway"] == "Home"].copy()              # Filtered to predict if only the home team will score 2 or more goals

df_home.shape
df_home.head()

,MatchID,Season,Date,Time,Team,Opponent,HomeAway,Goals,GoalsConceded,HTGoals,...,BMGMCA,BVCH,BVCD,BVCA,CLCH,CLCD,CLCA,LBCH,LBCD,LBCA
0,1,2021/2022,2021-08-13,20:00,Brentford,Arsenal,Home,2,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,2021/2022,2021-08-14,12:30,Man United,Leeds,Home,5,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3,2021/2022,2021-08-14,15:00,Burnley,Brighton,Home,1,2,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,4,2021/2022,2021-08-14,15:00,Chelsea,Crystal Palace,Home,3,0,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,5,2021/2022,2021-08-14,15:00,Everton,Southampton,Home,3,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
# Predictions are going to be made by getting the average goalsn shots and shots on target in the last 5 games. shift(1) ensures that there is no leakage because it guarentees each row only looks at the matches strictly before itself

df_home = df_home.sort_values(["Team", "Date"]).reset_index(drop=True)

def add_last5_home(g):
    g = g.sort_values("Date").copy()
    for stat in ["Goals", "HS", "HST"]:
        g[f"{stat}_avg5"] = g[stat].shift(1).rolling(5, min_periods=1).mean()
    return g

df_home = (
    df_home.groupby("Team", group_keys=False)
           .apply(add_last5_home)
           .reset_index(drop=True))
    
df_home["Target"] = (df_home["Goals"] >= 2).astype(int)

In [3]:
model_cols = ["MatchID", "Date","Time", "Team", "Opponent", "HomeAway","Target","Goals_avg5", "HS_avg5", "HST_avg5",]

df_model = df_home[model_cols].copy()
df_model.sort_values(["Date", "Time"]).head()

,MatchID,Date,Time,Team,Opponent,HomeAway,Target,Goals_avg5,HS_avg5,HST_avg5
266,1,2021-08-13,20:00,Brentford,Arsenal,Home,1,NaN,NaN,NaN
1216,2,2021-08-14,12:30,Man United,Leeds,Home,1,NaN,NaN,NaN
456,3,2021-08-14,15:00,Burnley,Brighton,Home,0,NaN,NaN,NaN
513,4,2021-08-14,15:00,Chelsea,Crystal Palace,Home,1,NaN,NaN,NaN
703,5,2021-08-14,15:00,Everton,Southampton,Home,1,NaN,NaN,NaN


In [4]:
#The split is done by date to ensure no leakage
train_end = "2024-08-01"
val_end = "2025-08-01"

train = df_model[df_model["Date"] < train_end]

validation = df_model[(df_model["Date"] >= train_end) & 
                     (df_model["Date"] < val_end)]

test = df_model[df_model["Date"] >= val_end]

print(train.shape, validation.shape, test.shape)

(1140, 10) (380, 10) (380, 10)


In [5]:
print(len(set(train["MatchID"]) & set(validation["MatchID"])))
print(len(set(validation["MatchID"]) & set(test["MatchID"])))
print(len(set(train["MatchID"]) & set(test["MatchID"])))         # If all are equal to zero then theres no  overlap and hence no leakage

0
0
0


In [6]:
print(len(train) + len(validation) + len(test) == len(df_model))

True


In [7]:
total_matches = len(df_model)
print(f"Train percentage:      {len(train) / total_matches:.1%}")
print(f"Validation percentage: {len(validation) / total_matches:.1%}")
print(f"Test percentage:       {len(test) / total_matches:.1%}")

Train percentage:      60.0%
Validation percentage: 20.0%
Test percentage:       20.0%


In [8]:
feature_cols = ["Goals_avg5", "HS_avg5", "HST_avg5"]

X_train = train[feature_cols]
y_train = train["Target"]
X_val   = validation[feature_cols]
y_val = validation["Target"]
X_test  = test[feature_cols]
y_test = test["Target"]